In [239]:
import pandas as pd
pd.set_option("mode.copy_on_write", True)
import numpy as np
from typing import no_type_check, Set, Sequence, Any,Optional,List,Callable,Dict,Union
import networkx as nx
import itertools
from collections import defaultdict
import sys
import os
from pathlib import Path
sys.path.append("../../spannerlib/spannerlib")

import time


from spannerlib.span import Span
from spannerlib.data_types import (
    Var, 
    FreeVar, 
    RelationDefinition, 
    Relation, 
    IEFunction,
    AGGFunction,
    IERelation, 
    Rule, 
    pretty
)
from spannerlib.ra import (
    _col_names,
    get_const,
    select,
    project,
    rename,
    union,
    intersection,
    difference,
    join,
    product,
    groupby,
    ie_map,
    merge_rows
)

from spannerlib.term_graph import graph_compose, merge_term_graphs_pair,rule_to_graph,add_relation,add_project_uniq_free_vars
from spannerlib.engine import Engine, IEFunction, AGGFunction
from spannerlib import get_magic_session,Span


from graph_rewrite import draw
from graph_rewrite import rewrite, rewrite_iter


import logging
logger = logging.getLogger(__name__)

## Utils ##

In [240]:
def schema_match(schema,expected,ignore_types=None):
    """checks if"""
    if len(schema) != len(expected):
        return False
    if ignore_types is None:
        ignore_types = []
    for x,y in zip(schema,expected):
        if x in ignore_types:
            continue
        if not issubclass(x,y):
            return False
    return True


def is_of_schema(relation,schema,ignore_types=None):
    """checks if a relation is of a given schema"""
    try:
        if len(relation) != len(schema):
            return False
        if ignore_types is None:
            ignore_types = []
        for x,y in zip(relation,schema):
            if type(x) in ignore_types:
                continue
            if not isinstance(x,y):
                return False
        return True
    except Exception as e:
        logger.error(f"Got Error when computing:\n"
                     f"is_of_scehma({relation},{schema})\n"
                     f"Error: {e}")
        raise e

def type_merge(type1,type2):
    if issubclass(type1,type2):
        return type1
    elif issubclass(type2,type1):
        return type2
    else:
        raise ValueError(f"Trying to merge types {type1},{type2}, types are incompatible")

def schema_merge(schema1,schema2):
    """merges two schemas, taking the stricter type between the two for each index"""
    if len(schema1) != len(schema2):
        raise ValueError(f"Trying to merge schemas {schema1},{schema2} schemas must be of the same length")
    
    new_schema = [type_merge(x,y) for x,y in zip(schema1,schema2)]
    return new_schema

In [241]:
import re
STRING_PATTERN = re.compile(r"^[^\r\n]+$")

def isFloat(s):  
   n = '0123456789.' 
   return (all(x in n for x in s) and s.count('.') == 1)  
 
def isInt(s):  
   n = '0123456789'    
   return all(x in n for x in s) 

def _infer_relation_schema(row) -> Sequence[type]: # Inferred type list of the given relation
    """
    Guess the relation type based on the data.
    We support both the actual types (e.g. 'Span'), and their string representation ( e.g. `"[0,8)"`).

    **@raise** ValueError: if there is a cell inside `row` of an illegal type.
    """
    relation_types = []
    for cell in row:
        if not isinstance(cell, str):
            relation_types.append(type(cell))
        elif isInt(cell):
            relation_types.append(int)
        elif isFloat(cell):
            relation_types.append(float)
        elif cell in ['True', 'False']:
            relation_types.append(bool)
        else:
            relation_types.append(str)
        
    return relation_types

In [242]:
class DB(dict):
    def __repr__(self):
        key_str=', '.join(self.keys())
        return f'DB({key_str})'

In [243]:
def _col_names(length):
    # these names wont conflict with logical variables since they must always start with Uppercase letters
    return [f'col_{i}' for i in range(length)]

In [244]:

# some select theta functions

class equalConstTheta():
    def __init__(self,*pos_val_tuples):
        self.pos_val_tuples = pos_val_tuples
    def __call__(self,df):
        masks = [df.iloc[:,pos]==val for pos,val in self.pos_val_tuples]
        return pd.concat(masks,axis=1).all(axis=1)
    def __str__(self):
        return f'''Theta({', '.join([f'col_{pos}={val}' for pos,val in self.pos_val_tuples])})'''
    def __repr__(self):
        return str(self)
    def __eq__(self,other):
        if not isinstance(other,equalConstTheta):
            return False
        return self.pos_val_tuples == other.pos_val_tuples

class equalColTheta():
    def __init__(self,*col_pos_tuples):
        self.col_pos_tuples = col_pos_tuples

    def __call__(self,df):
        masks = [df.iloc[:,pos1]==df.iloc[:,pos2] for pos1,pos2 in self.col_pos_tuples]
        return pd.concat(masks,axis=1).all(axis=1)    
    def __str__(self):
        return f'''Theta({', '.join([f'col_{pos1}=col_{pos2}' for pos1,pos2 in self.col_pos_tuples])})'''
    def __repr__(self):
        return str(self)
    def __eq__(self,other):
        if not isinstance(other,equalColTheta):
            return False
        return self.col_pos_tuples == other.col_pos_tuples

In [245]:
def get_const(const_dict,**kwargs):
    return pd.DataFrame([const_dict])


def is_truthy(df):
    return df.shape==(1,0)

def is_falsy(df):
    return df.shape==(0,0)

In [246]:
def select(df,theta,schema,**kwargs):
    if df is None or df.empty:
        return pd.DataFrame(columns=schema)
    if callable(theta):
        return df[theta(df)]
    else:
        raise ValueError(f"theta must be callable, got {theta}")

def project(df,schema,**kwargs):
    if df is None or df.empty:
        return pd.DataFrame(columns=schema)
    return df[schema]
    
def rename(df,schema,**kwargs):
    if df is None or df.empty:
        return pd.DataFrame(columns=schema)
    
    df=df.copy()
    df.columns = schema
    return df

def intersection(df1,df2,schema,**kwargs):
    if df1 is None or df2 is None or df1.empty or df2.empty:
        return pd.DataFrame(columns=schema)
    return pd.merge(df1,df2,how='inner',on=list(df1.columns))

def difference(df1,df2,schema,**kwargs):
    if df1 is None or df2 is None or df1.empty or df2.empty:
        return pd.DataFrame(columns=schema)
    return pd.concat([df1,df2]).drop_duplicates(keep=False)


def product(df1,df2,schema,**kwargs):
    if df1 is None or df2 is None or df1.empty or df2.empty:
        return pd.DataFrame(columns=schema)
    return pd.merge(df1,df2,how='cross')

def join(df1,df2,schema,**kwargs):
    if df1 is None or df2 is None or is_falsy(df1) or is_falsy(df2):
        return pd.DataFrame(columns=schema)

    # if one of the dataframes is truthy, return the other
    # this solves the problem of joining with a constant
    if is_truthy(df1):
        return df2
    if is_truthy(df2):
        return df1

    cols1 = set(df1.columns)
    cols2 = set(df2.columns)
    on = cols1 & cols2
    # get only logical variables
    # on = [ col for col in on if isinstance(col,str) and col[0].isupper()]
    on = list(on)
    if len(on)==0:
        return pd.merge(df1,df2,how='cross')
    else:
        return pd.merge(df1,df2,how='inner',on=on)
    
def merge_rows(*dfs):
    return pd.DataFrame(
        set.union(*[set(df.itertuples(index=False,name=None)) for df in dfs])
    )


def union(*dfs,schema,**kwargs):
    # use numpy arrays to ignore column names
    non_empty_dfs = []
    for df in dfs:
        if df is not None and not df.empty:
            non_empty_dfs.append(df)
    if len(non_empty_dfs)==0:
        return pd.DataFrame(columns=schema)
    else:
        return rename(merge_rows(*non_empty_dfs),schema)
        # This line didnt work since drop duplicates doesnt work correctly on non primitive classes such as Spans
        # return pd.DataFrame(np.concatenate(non_empty_dfs,axis=0),columns=schema).drop_duplicates(ignore_index=True)
        
def groupby(df,schema,agg,**kwargs):
    if df is None or df.empty:
        return pd.DataFrame(columns=schema)
    
    # rename columns to numbers so that we can aggregate the same free var to multiple places
    uniq_cols_df = rename(df,schema=[i for i in range(len(schema))])

    groupby_cols = [i for i,agg_func in enumerate(agg) if agg_func is None]
    agg_by_cols = {i:agg_func for i,agg_func in enumerate(agg) if agg_func is not None}
    # a real groupby
    if len(groupby_cols)>0:
        return rename(
            project(
                uniq_cols_df.groupby(groupby_cols).agg(agg_by_cols).reset_index(),
                schema = uniq_cols_df.columns
                ),
            schema)
    # no group by vars, so aggs contain all columns and schema simply orders them
    else:

        # this conversion magic is caused by an inconsistency between series and dataframes aggs,
        # to enable using both function and str aliases we
        # we take each column, convert to a frame
        # aggregate it and then squeeze it to a series (which has a single value)
        # then feed that to the dataframe constructor
        return rename(
            pd.DataFrame({
                col:[uniq_cols_df[col].to_frame().agg(agg_by_cols[col]).squeeze()] for col in range(len(agg_by_cols))
            }),
            schema)

In [247]:
def coerce_tuple_like(name,func,input,output):
    if isinstance(output,(tuple,list)):
        return output
    else:
        logger.debug(f"IEFunction {name} with underlying function {func}\n"
                        f"returned a value that is not a tuple/list\n"
                        f"for input {input} -> {output}\n"
                        f"coercing to tuple")
        return (output,)

def assert_ie_schema(name,func,value,expected_schema,arity,input_or_output='input'):
    if callable(expected_schema):
        expected_schema = expected_schema(arity)
    if not is_of_schema(value,expected_schema):
        raise ValueError(
            f"IEFunction {name} with underlying function {func}\n"
            f"received an {input_or_output} value {value}(schema={pretty(_infer_relation_schema(value))})\n"
            f"but expected {pretty(expected_schema)}")

def assert_iterable(name,func,input,output):
    try:
        out_iter = iter(output)
    except TypeError:
        raise ValueError(f"IEFunction {name} with underlying function {func}\n"
                f"returned a value that is not an iterable\n"
                f"for input {input} -> {output}")

def map_iter(df,name,func,in_schema,out_schema,in_arity,out_arity,**kwargs):
    """helper function returns an iterator that applies a function to each row of a dataframe
    """
    for _,in_row in df.iterrows():
        in_row = list(in_row)
        assert_ie_schema(name,func,in_row,in_schema,in_arity,input_or_output='input')
        output = func(*in_row)
        assert_iterable(name,func,in_row,output)
        for out_row in output:
            out_row = coerce_tuple_like(name,func,in_row,out_row)
            out_row = list(out_row)
            assert_ie_schema(name,func,out_row,out_schema,out_arity,input_or_output='output')
            yield in_row + out_row

def ie_map(df,name,func,in_schema,out_schema,in_arity,out_arity,**kwargs):
    """given an indexed dataframe, apply an ie function to each row and return the output 
    such that each output relation is indexed by the same index as the input relation that generated it
    """
    if df is None or df.empty:
        return pd.DataFrame(columns=_col_names(in_arity+out_arity))
    output_iter = map_iter(df,name,func,in_schema,out_schema,in_arity,out_arity)
    total_arity = in_arity + out_arity
    return pd.DataFrame(output_iter,columns=_col_names(total_arity))

In [248]:

def get_rel(rel,db,**kwargs):
    # helper function to get the relation from the db for external relations
    return db[rel]

op_to_func = {
    'union':union,
    'intersection':intersection,
    'difference':difference,
    'select':select,
    'project':project,
    'rename':rename,
    'join':join,
    'ie_map':ie_map,
    'get_rel':get_rel,
    'get_const':get_const,
    'product':product,
    'groupby':groupby
}

In [249]:
def _in_cycle(g):
    return list(set(
        itertools.chain.from_iterable(nx.cycles.simple_cycles(g))
    ))

def _depends_on_cycle(g):
    in_cycle_nodes = _in_cycle(g)
    depends_on_cycle = {
        node for node in g.nodes if node in in_cycle_nodes or 
        len(set(nx.descendants(g,node)).intersection(in_cycle_nodes))>0
    }
    return depends_on_cycle

In [250]:
def calculate_dfs_rows(*dfs):
    if dfs:
        return sum([df.shape[0] for df in dfs])
    return 0

In [251]:
def profile_wrapper(op_func, profile_data, children_results, u_data):
    start = time.time()
    res = op_func(*children_results, **u_data)
    end = time.time()
    profile_data[op_func.__name__]["row_count"] += calculate_dfs_rows(*children_results)
    profile_data[op_func.__name__]["total_time"] += end - start
    return res, end - start

In [252]:
def _collect_children_and_run(G,u,results,profile_data,stack,log=False):
    children = list(G.successors(u))
    u_data = G.nodes[u]

    children_results = [results[v][-1] for v in children]
    op_func = op_to_func[u_data['op']]

    if log:
        logger.debug(f"computing node {u} with children {children} and data {u_data} , stack = {stack}")
        logger.debug(f"children results are {children_results}")
        logger.debug(f"children_data is {[G.nodes[v] for v in children]}")
    try:
        res, op_time = profile_wrapper(op_func, profile_data, children_results, u_data)
    except Exception as e:
        raise Exception(f'During excution of node {u} with args {children_results} and kwargs {u_data}'
                        f' got error {e}'
        )
    if log:
        logger.debug(f"result of node {u} is {res}")
    results[u].append(res)
    G.nodes[u]['op_time'] = op_time
    return res


In [253]:
def compute_acyclic_node(G,u,results,profile_data,stack=None,log=False):
    res = _collect_children_and_run(G,u,results,profile_data,[],log=log)
    logger.debug(f"setting {u} to final since it is acyclic\n")
    G.nodes[u]['final'] = True
    return res

def compute_recursive_node(G,u,results,profile_data,stack=None,log=False):

    if stack is None:
        stack = []

    children = list(G.successors(u))
    u_data = G.nodes[u]
    op_func = op_to_func[u_data['op']]

    if u_data.get('final',False):
        return results[u][-1]    


    logger.debug(f"computing node {u} with stack {stack}")


    went_in_a_cycle = u in stack
    if went_in_a_cycle:
        logger.debug(f"went in a cycle at {u}, computing op with empty children if necessary\n")
        # for each child that doesnt have data, put an empty df instead of it
        res = _collect_children_and_run(G,u,results,profile_data,stack,log=log)
        return res

    # if we are here we are in a cycle but didnt return to an old position yet
    # then we compute all our children first
    for v in children:
        stack.append(u)
        compute_recursive_node(G,v,results,profile_data,stack,log=log)
        stack.pop()

    # compute and mark as final if reached fixed point
    res = _collect_children_and_run(G,u,results,profile_data,stack,log=log)

    all_children_final = all(G.nodes[v].get('final',False) for v in children)
    fixed_point_reached = len(results[u])>1 and results[u][-1].equals(results[u][-2])

    if all_children_final:
        logger.debug(f"setting {u} to final since all children are final\n")
        G.nodes[u]['final'] = True
    elif fixed_point_reached:
        logger.debug(f"setting {u} to final since fixed point has been achieved\n")
        # if u==9:
        #     logger.debug(f"graph nodes are{g.nodes(data=True)}")
        G.nodes[u]['final'] = True
    else:
        logger.debug(f"{u} not final yet so we will need to run another iteration\n")

    return res

def compute_node(G,root,ret_inter=False,log=False):

    # makes sure there is always a last value in the list for each key
    # which is None
    list_with_none_factory = lambda : [None]
    results_dict = defaultdict(list_with_none_factory)
    profile_data = defaultdict(lambda: {"row_count": 0, "total_time": 0.0})
    start_time = time.time()
    depends_on_cycle = _depends_on_cycle(G)
    not_depends_on_cycle = [u for u in G.nodes if u not in depends_on_cycle]

    # compute non cyclic nodes in postorder
    non_cycle_topological_sort = list(nx.topological_sort(nx
                                                          .DiGraph(nx.subgraph(G,not_depends_on_cycle))))
    for u in non_cycle_topological_sort[::-1]:
        compute_acyclic_node(G,u,results_dict,profile_data,log=log)

    logger.debug(f"the following nodes were computed non cyclically {non_cycle_topological_sort}")
    # now that all initial conditions for recursions are set
    # run the compute_recursive_node on u
    logger.debug(f"running compute_recursive_node on {root}")

    while True:
        res = compute_recursive_node(G,root,results_dict,profile_data,log=log)
        if G.nodes[root].get('final',False):
            break
    end_time = time.time()
    profile_data['total_time'] = end_time - start_time
    if ret_inter:
        return res,profile_data, results_dict
    else:
        return res, profile_data


In [254]:
graph  = nx.DiGraph()
graph.add_nodes_from([
    0,1,2,3,
])
graph.add_edges_from(
    [(0,1),(0,2),(1,3),(2,3),(3,4)]
)
draw(graph)
edges_df = pd.DataFrame(list(graph.edges),columns=['S','T'])
edges_df
db = DB({
    'edges':edges_df
})

In [255]:
g = nx.DiGraph()
g.add_nodes_from([
    ('edges',{'rel':'edges','op':'get_rel','db':db}),
    (1,{'op':'rename','schema':['S','T']}),
    (2,{'op':'rename','schema':['S','X']}),
    (3,{'op':'rename','schema':['X','T']}),
    (4,{'op':'join','schema':['S','X','T']}),
    (5,{'op':'project','schema':['S','T']}),
    ('reachable',{'op':'union','schema':[0,1]}),
    (6,{'op':'rename','schema':['S','T']})]
)
g.add_edges_from([
    (1,'edges'),
    (2,'edges'),
    (4,2),
    (4,3),
    (5,4),
    ('reachable',5),
    ('reachable',1),
    (3,'reachable'),
    (6,'reachable')
])
draw(g)

In [256]:
#res, profile_data, ret_inter = compute_node(g,6,ret_inter=True)
#profile_df = pd.DataFrame(profile_data).T
#profile_df

In [257]:
def func(str):
    yield (len(str),)
    
G = nx.DiGraph()

str_list = ["a" * n for n in range(20000)]

db = DB({
    "string": pd.DataFrame({"col_0": str_list}),
    "string_length": pd.DataFrame(columns=["col_0", "col_1"])  
})


nodes = {
    "string": {
        "op": "get_rel",
        "rel": "string",
        "rule_id": "{0, 'fact'}",
        "schema": ["col_0"],
        "db": db
    },
    "string_length": {
        "op": "union",
        "rel": "string_length",
        "rule_id": "{0, 'fact'}",
        "schema": ["col_0", "col_1"]
    },
    "0": {
        "op": "rename",
        "schema": ["Str"],
        "rule_id": "{0}"
    },
    "1": {
        "op": "project",
        "schema": ["Str"],
        "rule_id": "{0}"
    },
    "2": {
        "op": "project",
        "schema": ["Str"],
        "rule_id": "{0}"
    },
    "3": {
        "op": "ie_map",
        "func": func,
        "in_arity": 1,
        "out_arity": 1,
        "schema": ["col_0", "col_1"],
        "rule_id": "{0}",
        "name": "Length",
        "in_schema": [str],
        "out_schema": [int]
    },
    "4": {
        "op": "rename",
        "schema": ["Str", "Len"],
        "rule_id": "{0}"
    },
    "5": {
        "op": "rename",
        "schema": ["Str", "Len"],
        "rule_id": "{0}"
    },
    "6": {
        "op": "project",
        "schema": ["Str", "Len"],
        "rule_id": "{0}"
    },
    "7": {
        "op": "join",
        "schema": ["Str", "Len"],
        "rule_id": "{0}"
    },
    "8": {
        "op": "project",
        "schema": ["Str", "Len"],
        "rel": "_string_length_0",
        "rule_id": "{0}"
    },
    "9": {
        "op": "rename",
        "schema": ["Str", "Len"]
    },
    "10": {
        "op": "project",
        "schema": ["Str", "Len"]
    }
}

# Add nodes to the graph with their attributes
for node, attrs in nodes.items():
    G.add_node(node, **attrs)

# Define edges as per the Mermaid diagram structure
edges = [
    ("0", "string"),
    ("1", "0"),
    ("2", "1"),
    ("3", "2"),
    ("4", "3"),
    ("5", "4"),
    ("6", "5"),
    ("7", "6"),
    ("7", "1"),
    ("string_length", "8"),
    ("8", "7"),
    ("9", "string_length"),
    ("10", "9")
]

# Add edges to the graph
G.add_edges_from(edges)
draw(G)

In [258]:
#db["string"]


In [259]:
#res, profile_data = compute_node(G, "10")
#print(res.sort_values(by="Len"))
#profile_df = pd.DataFrame(profile_data).T
#profile_df


# Query I - tax year 

In [260]:
from spannerlib.ie_func.basic import rgx

In [261]:
magic_session = get_magic_session()

In [262]:

def rgx_is_match(pattern, # the delimeter pattern to split on
    text, # the text to be split, can be either string or Span
    ):
    """
    An IE function which given a delimeter rgx pattern and a text, 
    returns True if any match is found, False otherwise.
    """
    for _ in rgx(pattern,text):
        return [("True", )]
    return []

magic_session.register('rgx_is_match',rgx_is_match,in_schema=[str,str],out_schema=[str])

In [263]:
magic_session.import_rel('sms_rel','sms_rel.csv')

In [264]:
%%spannerlog
sms_tax_year(sms_id,sender_id,is_sender_in_contacts,sms_body,sms_date,is_spam)<-
    sms_rel(sms_id,sender_id,is_sender_in_contacts,sms_body,sms_date,is_spam),
    rgx_is_match('refund for\s*([1-2][0-9]{3})',sms_body)->(res).

sms_tax_year(sms_id,sender_id,is_sender_in_contacts,sms_body,sms_date,is_spam)<-
    sms_rel(sms_id,sender_id,is_sender_in_contacts,sms_body,sms_date,is_spam),
    rgx_is_match('([1-2][0-9]{3})[a-zA-z ]*open for[a-zA-Z ]*refund',sms_body)->(res).


In [265]:
magic_session.export('?sms_tax_year(sms_id,sender_id,is_sender_in_contacts,sms_body,sms_date,is_spam)')

,sms_id,sender_id,is_sender_in_contacts,sms_body,sms_date,is_spam
0,S123456A789010,TaxReturn,FALSE,"Nadav RD 15, According to our records, you are...","05/01/2023, 20:27",TRUE
1,S123456A789011,033-030571,FALSE,"Hi Nadav army, The year 2024 has arrived, and ...","13/06/2024, 12:02",TRUE
2,S123456A789013,RefundTax,FALSE,Check if you are eligible for a tax refund for...,"29/11/2021, 18:12",TRUE
3,S223456A787614,072-3491746,FALSE,Your tax refund for 2018 may be higher than yo...,"05/01/2024, 16:00",TRUE
4,S234567A890121,TaxReturn,FALSE,"Nadav RD 15, According to our records, you are...","22/01/2023, 19:57",TRUE
5,S234567A890122,033-030571,FALSE,"Hi Nadav army, The year 2024 has arrived, and ...","13/06/2024, 10:14",TRUE
6,S234567A890124,MyReTax,FALSE,Check your eligibility for a tax refund for 20...,"21/05/2022, 20:39",TRUE
7,S234567A890127,072-3491746,FALSE,Check if you qualify for a tax refund for 2023...,"07/02/2024, 08:25",TRUE
8,S345678A901233,033-030571,FALSE,"Hi Nadav army, The year 2024 has arrived, and ...","03/07/2024, 13:40",TRUE
9,S352109A847627,FreeTax,FALSE,Find out how much your tax refund for 2020 is ...,"06/01/2024, 13:25",TRUE


In [266]:
graph,root = magic_session.export('?sms_tax_year(sms_id,sender_id,is_sender_in_contacts,sms_body,sms_date,is_spam)',plan_query=True,draw_query=True)

In [267]:
res, profile_data = compute_node(graph,root)

# Relational Optimizations

In [268]:
# We start by removing all unnecesary project nodes - all projects here that are not childs of an product nodes are redundant
def is_not_product(match,node):
    if 'product' in match[node]['op']:
        return False
    return True
            
rewrite(graph, lhs='project_node[op="project", schema];x[op]->project_node->y', p='x[op],y', rhs='x[op]->y',condition=lambda match:
    is_not_product(match, "x") and is_not_product(match, "y"), is_recursive=True)

draw(graph)

In [269]:
# Every rename's schema can be propagated to the child node
def schema(match, node):
    return match[node]['schema']
rewrite(graph, lhs='y[op="rename",schema]->z[schema];x->y', p='x,z[schema]', rhs='x->z[schema={{y_schema}}]',
        render_rhs={'y_schema': lambda match: schema(match, 'y')}, is_recursive=True)
draw(graph)

In [270]:
# move the union to be before the join,join node, and merge both projects that project sms_body into a single project node
for match in rewrite_iter(graph,
        lhs='union_node[op="union",schema]->join_node[op="join",schema]->ie_node[func,schema],join_node->sms_rel',
        p='union_node[op],ie_node[func,schema],sms_rel',
        rhs='union_node[op,schema={{ie_node_schema}}]->ie_node[func,schema], sms_rel', 
        render_rhs={'ie_node_schema': lambda match: schema(match, 'ie_node'), 'join_node_schema': lambda match: schema(match, 'join_node')}):
        join_schema = match['join_node']['schema']
rewrite(graph, lhs='project_node[op="project"]->union_node[op="union"], sms_rel[op="get_rel",schema]', 
        p='project_node[op],sms_rel[op,schema],union_node[op]',
        rhs='project_node[op]->join_node[op="join",schema={{join_schema}}]->union_node[op], join_node->sms_rel[op,schema]',
        render_rhs={'join_schema': lambda match: join_schema})
rewrite(graph, lhs='x->project_node_1[op="project"]->sms_rel[op="get_rel"], y->project_node_2[op="project"]->sms_rel', 
        p='project_node_1[op],project_node_2[op],sms_rel[op],x,y',
        rhs='x->project_node_1&project_node_2[op]->sms_rel[op],y->project_node_1&project_node_2', is_recursive=True)
draw(graph)

In [271]:
sms_rel = pd.read_csv('sms_rel.csv')
graph.nodes['sms_rel']['db'] = DB({'sms_rel': sms_rel, 'sms_tax_year': pd.DataFrame()})
before_optimization_profile_df = pd.DataFrame(profile_data).T
print("Before optimization:")
print(before_optimization_profile_df)
res, profile_data = compute_node(graph,root)
profile_df = pd.DataFrame(profile_data).T
print("After optimization:")
print(profile_df)
res

Before optimization:
              row_count  total_time
get_rel        0.000000    0.000002
rename       634.000000    0.002511
project     1794.000000    0.008577
get_const      0.000000    0.000787
product      582.000000    0.007478
ie_map       580.000000    0.045061
join         598.000000    0.003392
union         18.000000    0.001128
total_time     0.070166    0.070166
After optimization:
             row_count  total_time
get_rel       0.000000    0.000004
project     885.000000    0.002807
get_const     0.000000    0.000412
product     580.000000    0.005167
ie_map      578.000000    0.046071
union        18.000000    0.000786
join        307.000000    0.001980
total_time    0.058013    0.058013


,sms_id,sender_id,is_sender_in_contacts,sms_body,sms_date,is_spam
0,S123456A789010,TaxReturn,False,"Nadav RD 15, According to our records, you are...","05/01/2023, 20:27",True
1,S234567A890121,TaxReturn,False,"Nadav RD 15, According to our records, you are...","22/01/2023, 19:57",True
2,S456789A012343,TaxReturn,False,"Stav Aviram - Army, According to our records, ...","22/01/2023, 11:32",True
3,S678901A234565,033-030413,False,This is the last chance to withdraw. If you ha...,"10/09/2024, 10:34",True
4,S789012A345676,Missim,False,You might also receive such a message! Hello S...,"06/08/2024, 15:52",True
5,S890123A456787,TaxBack,False,We noticed that you might be eligible for a ta...,"28/08/2024, 10:07",True
6,S901234A567898,Missim,False,You might also receive such a message! Hello S...,"22/08/2024, 14:06",True
7,S123456A789011,033-030571,False,"Hi Nadav army, The year 2024 has arrived, and ...","13/06/2024, 12:02",True
8,S234567A890122,033-030571,False,"Hi Nadav army, The year 2024 has arrived, and ...","13/06/2024, 10:14",True
9,S345678A901233,033-030571,False,"Hi Nadav army, The year 2024 has arrived, and ...","03/07/2024, 13:40",True


# Optimization I - subregex

In [272]:
year_dict = {"_C0":'[1-2][0-9]{3}'}

rewrite(graph, lhs='project_node[op="project"];product_node[op="product"]->project_node',
        p='product_node[op],project_node[op]',
        rhs='new_ie_node[op="ie_map",func={{func}},in_arity=2,out_arity=1,schema={{ie_schema}},name="rgx_is_match",in_schema={{in_schema}},out_schema={{out_schema}}]\
            ,new_product_node->new_get_const_node[op="get_const",const_dict={{const_dict}},schema={{const_schema}}],\
            new_product_node[op="product",schema={{product_schema}}]->project_node[op],\
            new_project_node_2[op="project",schema={{project_schema_2}}]->new_rename_node[op="rename",schema={{ie_schema}}]->new_ie_node\
            ->new_project_node_1[op="project",schema={{project_schema_1}}]->new_product_node,\
            product_node[op]->new_project_node_2',
        render_rhs={'func': lambda match: rgx_is_match,
                    'ie_schema': lambda match: ['_F0', 'sms_body', 'res'],
                    'product_schema': lambda match: ['sms_body', '_C0'],
                    'const_schema': lambda match: ['_C0'],
                    'project_schema_1': lambda match: ['_C0', 'sms_body'],
                    'project_schema_2': lambda match: ['sms_body'],
                    'const_dict': lambda match: year_dict,
                    'in_schema': lambda match: [str,str],
                    'out_schema': lambda match: [str]})

draw(graph)

In [273]:
graph.nodes['sms_rel']['db'] = DB({'sms_rel': sms_rel, 'sms_tax_year': pd.DataFrame()})
print("Profile data before optimization")
print(profile_df)
res, profile_data = compute_node(graph, root)
print("Profile data after optimization")
profile_df = pd.DataFrame(profile_data).T
print(profile_df)
res


Profile data before optimization
             row_count  total_time
get_rel       0.000000    0.000004
project     885.000000    0.002807
get_const     0.000000    0.000412
product     580.000000    0.005167
ie_map      578.000000    0.046071
union        18.000000    0.000786
join        307.000000    0.001980
total_time    0.058013    0.058013
Profile data after optimization
             row_count  total_time
get_rel       0.000000    0.000003
project     827.000000    0.005103
get_const     0.000000    0.000902
product     446.000000    0.007211
ie_map      443.000000    0.040721
rename       77.000000    0.000137
union        18.000000    0.000790
join        307.000000    0.001930
total_time    0.057734    0.057734


,sms_id,sender_id,is_sender_in_contacts,sms_body,sms_date,is_spam
0,S123456A789010,TaxReturn,False,"Nadav RD 15, According to our records, you are...","05/01/2023, 20:27",True
1,S234567A890121,TaxReturn,False,"Nadav RD 15, According to our records, you are...","22/01/2023, 19:57",True
2,S456789A012343,TaxReturn,False,"Stav Aviram - Army, According to our records, ...","22/01/2023, 11:32",True
3,S678901A234565,033-030413,False,This is the last chance to withdraw. If you ha...,"10/09/2024, 10:34",True
4,S789012A345676,Missim,False,You might also receive such a message! Hello S...,"06/08/2024, 15:52",True
5,S890123A456787,TaxBack,False,We noticed that you might be eligible for a ta...,"28/08/2024, 10:07",True
6,S901234A567898,Missim,False,You might also receive such a message! Hello S...,"22/08/2024, 14:06",True
7,S123456A789011,033-030571,False,"Hi Nadav army, The year 2024 has arrived, and ...","13/06/2024, 12:02",True
8,S234567A890122,033-030571,False,"Hi Nadav army, The year 2024 has arrived, and ...","13/06/2024, 10:14",True
9,S345678A901233,033-030571,False,"Hi Nadav army, The year 2024 has arrived, and ...","03/07/2024, 13:40",True


# Query II - tax link

In [274]:
%%spannerlog
sms_tax_link(sms_id,sender_id,is_sender_in_contacts,sms_body,sms_date,is_spam)<-
sms_rel(sms_id,sender_id,is_sender_in_contacts,sms_body,sms_date,is_spam),
rgx_is_match('http.*[rR][eE][fF][uU][nN][dD].*\.com',sms_body)->(res).

sms_tax_link(sms_id,sender_id,is_sender_in_contacts,sms_body,sms_date,is_spam)<-
sms_rel(sms_id,sender_id,is_sender_in_contacts,sms_body,sms_date,is_spam),
rgx_is_match('http.*[tT][aA][xX].*\.com',sms_body)->(res).

In [275]:
graph, root = magic_session.export('?sms_tax_link(sms_id,sender_id,is_sender_in_contacts,sms_body,sms_date,is_spam)', plan_query=True, draw_query=True)

In [276]:
res, profile_data = compute_node(graph, root)
profile_df = pd.DataFrame(profile_data).T
print(profile_df)
res

              row_count  total_time
get_rel        0.000000    0.000004
rename       773.000000    0.001575
project     1933.000000    0.012517
get_const      0.000000    0.001184
product      582.000000    0.008112
ie_map       580.000000    0.043342
join         649.000000    0.004755
union         69.000000    0.001541
total_time     0.074712    0.074712


,sms_id,sender_id,is_sender_in_contacts,sms_body,sms_date,is_spam
0,S234567A890125,FastRefunds,FALSE,Your eligibility for a refund is pending. Clic...,"10/01/2024, 11:20",TRUE
1,S012345A671112,033-0129472,FALSE,Get your maximum tax refund guaranteed! Click ...,"04/01/2024, 14:45",TRUE
2,S678901A234569,033-0129472,FALSE,Free tax refund eligibility check available no...,"14/01/2024, 12:05",TRUE
3,S789012A345676,Missim,FALSE,You might also receive such a message! Hello S...,"06/08/2024, 15:52",TRUE
4,S789012A345682,FastRefunds,FALSE,"Hello Stav Combinatorics Partner, our records ...","12/02/2024, 09:05",TRUE
5,S901234A567902,SmartRefund,FALSE,Important! Check your tax refund eligibility f...,"17/01/2024, 10:10",TRUE
6,S123456A789017,FastRefunds,FALSE,Your eligibility for a refund is pending. Clic...,"02/02/2024, 12:15",TRUE
7,S352109A847628,FastRefunds,FALSE,"Stav Compi Rep, our records show your tax refu...","20/01/2024, 15:20",TRUE
8,S352109A847627,FreeTax,FALSE,Find out how much your tax refund for 2020 is ...,"06/01/2024, 13:25",TRUE
9,S789012A345681,TaxEasy,FALSE,Get a full refund estimation in minutes - most...,"29/01/2024, 15:45",TRUE


# Query III - money amount

In [277]:
%%spannerlog
sms_money(sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam) <-
    sms_rel(sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam),
    rgx_is_match('\d{1,3},\d{3}', sms_body) -> (res).

sms_money(sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam) <-
    sms_rel(sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam),
    rgx_is_match('([nN][iI][sS])', sms_body) -> (res).

sms_money(sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam) <-
    sms_rel(sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam),
    rgx_is_match('\d{1,3}[kK]', sms_body) -> (res).

In [278]:
graph, root = magic_session.export('?sms_money(sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam)', plan_query=True, draw_query=True)

In [279]:
res, profile_data = compute_node(graph, root)
profile_df = pd.DataFrame(profile_data).T
print(profile_df)
res

              row_count  total_time
get_rel        0.000000    0.000002
rename      1144.000000    0.001638
project     2884.000000    0.011451
get_const      0.000000    0.002745
product      873.000000    0.007343
ie_map       870.000000    0.067192
join         972.000000    0.005131
union        102.000000    0.001968
total_time     0.098956    0.098956


,sms_id,sender_id,is_sender_in_contacts,sms_body,sms_date,is_spam
0,S235137A436136,Bit,FALSE,Amit CS Technion sent you money via bit! 30 NI...,"29/11/2021, 18:13",FALSE
1,S580806A532036,Bit,FALSE,Dani sent you money via bit! 110 NIS are waiti...,"01/03/2022, 12:33",FALSE
2,S789012A345676,Missim,FALSE,You might also receive such a message! Hello S...,"06/08/2024, 15:52",TRUE
3,S936463A791146,Bit,FALSE,Boaz sent you money via bit! 100 NIS are waiti...,"02/04/2024, 20:14",FALSE
4,S901234A567904,QuickRefund,FALSE,Stav Compi Rep - Up to 30K NIS in tax refunds ...,"14/02/2024, 13:25",TRUE
...,...,...,...,...,...,...
65,S476521A000764,Bit,FALSE,Mia sent you money via bit! 123 NIS are waitin...,"25/02/2024, 11:14",FALSE
66,S409002A166504,Refunds,FALSE,"Nadav Numeric Algorithms, our records show you...","02/12/2024, 14:31",TRUE
67,S890123A456792,FastRefunds,FALSE,"Stav Combinatorics Partner, dont miss out! Emp...","30/01/2024, 18:00",TRUE
68,S345678A901233,033-030571,FALSE,"Hi Nadav army, The year 2024 has arrived, and ...","03/07/2024, 13:40",TRUE


# Query IV - wrong receiver

In [280]:
truecaller_receiver_names = [
        'Stav My Love',
        'Stav CS Technion',
        'Stav Combinatorics Partner',
        'Stav Aviram - Army',
        'Stav Compi',
        'Stav Compi Rep'
        ]

In [281]:
%%spannerlog
sms_wrong_receiver(sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam) <-
    sms_rel(sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam),
    rgx_is_match('(?i)\\bhi\\s(Stav My Love|Stav CS Technion|Stav Combinatorics Partner|Stav Aviram - Army|Stav Compi|Stav Compi Rep)\\b.*?[.!?\\n]', sms_body) -> (res).

sms_wrong_receiver(sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam) <-
    sms_rel(sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam),
    rgx_is_match('(?i)\\bhello\\s(Stav My Love|Stav CS Technion|Stav Combinatorics Partner|Stav Aviram - Army|Stav Compi|Stav Compi Rep)\\b.*?[.!?\\n]', sms_body) -> (res).

sms_wrong_receiver(sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam) <-
    sms_rel(sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam),
    rgx_is_match('(?i)(Stav My Love|Stav CS Technion|Stav Combinatorics Partner|Stav Aviram - Army|Stav Compi|Stav Compi Rep).*?[.!?\\n]', sms_body) -> (res).

In [282]:
graph, root = magic_session.export('?sms_wrong_receiver(sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam)', plan_query=True, draw_query=True)

In [283]:
res, profile_data = compute_node(graph, root)
profile_df = pd.DataFrame(profile_data).T
print(profile_df)
res

              row_count  total_time
get_rel        0.000000    0.000003
rename       984.000000    0.003526
project     2724.000000    0.010598
get_const      0.000000    0.001472
product      873.000000    0.005934
ie_map       870.000000    0.062541
join         908.000000    0.004430
union         38.000000    0.001080
total_time     0.091041    0.091041


,sms_id,sender_id,is_sender_in_contacts,sms_body,sms_date,is_spam
0,S890123A456790,QuickRefund,FALSE,Stav Aviram - Army you may qualify for a tax r...,"02/01/2024, 11:30",TRUE
1,S789012A345679,TaxService,FALSE,"Stav Compi, Check your eligibility for a tax r...","01/01/2024, 10:00",TRUE
2,S789012A345676,Missim,FALSE,You might also receive such a message! Hello S...,"06/08/2024, 15:52",TRUE
3,S456789A012344,Shevah-Tax,FALSE,"Hello Stav Compi Rep, property owners that hav...","11/08/2024, 10:32",TRUE
4,S789012A345682,FastRefunds,FALSE,"Hello Stav Combinatorics Partner, our records ...","12/02/2024, 09:05",TRUE
5,S901234A567904,QuickRefund,FALSE,Stav Compi Rep - Up to 30K NIS in tax refunds ...,"14/02/2024, 13:25",TRUE
6,S567890A123456,TAXES,FALSE,"Stav Combinatorics Partner, reminder for you t...","05/03/2023, 20:42",TRUE
7,S678901A234566,ShevahTax,FALSE,Stav Aviram - Army In connection with your inq...,"12/07/2023, 09:19",TRUE
8,S890123A456788,073-3489675,FALSE,"Hi Stav Compi, Final chance to get your tax re...","07/12/2023, 10:40",TRUE
9,S352109A847628,FastRefunds,FALSE,"Stav Compi Rep, our records show your tax refu...","20/01/2024, 15:20",TRUE


# Query V - suspicious sender

In [284]:
%%spannerlog
sms_suspicious_sender(sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam) <-
    sms_rel(sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam),
    rgx_is_match('\d{3}-\d+', sender_id) -> (res).

sms_suspicious_sender(sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam) <-
    sms_rel(sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam),
    rgx_is_match('[tT][aA][xX]', sender_id) -> (res).

sms_suspicious_sender(sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam) <-
    sms_rel(sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam),
    rgx_is_match('[rR][eE][fF][uU][nN][dD]', sender_id) -> (res).

In [285]:
graph, root = magic_session.export('?sms_suspicious_sender(sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam)', plan_query=True, draw_query=True)

In [286]:
res, profile_data = compute_node(graph, root)
profile_df = pd.DataFrame(profile_data).T
print(profile_df)
res

              row_count  total_time
get_rel        0.000000    0.000003
rename      1181.000000    0.001939
project     3187.000000    0.010983
get_const      0.000000    0.001587
product      873.000000    0.006591
ie_map       870.000000    0.062151
join         976.000000    0.005253
union        372.000000    0.003490
total_time     0.093297    0.093297


,sms_id,sender_id,is_sender_in_contacts,sms_body,sms_date,is_spam
0,S678901A234569,033-0129472,FALSE,Free tax refund eligibility check available no...,"14/01/2024, 12:05",TRUE
1,S012345A671112,033-0129472,FALSE,Get your maximum tax refund guaranteed! Click ...,"04/01/2024, 14:45",TRUE
2,S234567A890125,FastRefunds,FALSE,Your eligibility for a refund is pending. Clic...,"10/01/2024, 11:20",TRUE
3,S123456A789013,RefundTax,FALSE,Check if you are eligible for a tax refund for...,"29/11/2021, 18:12",TRUE
4,S352109A847626,033-030885,FALSE,Your 2017 tax refund is waiting! Check it out ...,"02/10/2023, 15:11",TRUE
...,...,...,...,...,...,...
94,S345678A901233,033-030571,FALSE,"Hi Nadav army, The year 2024 has arrived, and ...","03/07/2024, 13:40",TRUE
95,S552075A397447,TaxAlerts,FALSE,"Nadav, our records show you are eligible for a...","28/11/2024, 14:31",TRUE
96,S234567A890122,033-030571,FALSE,"Hi Nadav army, The year 2024 has arrived, and ...","13/06/2024, 10:14",TRUE
97,S123456A788432,TaxOffice15,FALSE,"Due to the situation, you are eligible for a t...","16/01/2024, 20:03",TRUE


# Putting it together

In [287]:
%%spannerlog
sms_tax(sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam) <-
    sms_tax_year(sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam).

sms_tax(sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam) <-
    sms_tax_link(sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam).

sms_tax(sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam) <-
    sms_money(sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam).

sms_tax(sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam) <-
    sms_wrong_receiver(sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam).

sms_tax(sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam) <-
    sms_suspicious_sender(sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam).

In [288]:
graph, root = magic_session.export('?sms_tax(sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam)', plan_query=True, draw_query=True)

In [289]:
res, profile_data = compute_node(graph, root)
profile_df = pd.DataFrame(profile_data).T
print(profile_df)
res

               row_count  total_time
get_rel         0.000000    0.000002
rename       4857.000000    0.006710
project     12943.000000    0.034812
get_const       0.000000    0.002115
product      3783.000000    0.020980
ie_map       3770.000000    0.258473
join         4103.000000    0.020842
union         879.000000    0.012318
total_time      0.361451    0.361451


,sms_id,sender_id,is_sender_in_contacts,sms_body,sms_date,is_spam
0,S234567A890125,FastRefunds,FALSE,Your eligibility for a refund is pending. Clic...,"10/01/2024, 11:20",TRUE
1,S936463A791146,Bit,FALSE,Boaz sent you money via bit! 100 NIS are waiti...,"02/04/2024, 20:14",FALSE
2,S123456A789013,RefundTax,FALSE,Check if you are eligible for a tax refund for...,"29/11/2021, 18:12",TRUE
3,S789012A345682,FastRefunds,FALSE,"Hello Stav Combinatorics Partner, our records ...","12/02/2024, 09:05",TRUE
4,S901234A567902,SmartRefund,FALSE,Important! Check your tax refund eligibility f...,"17/01/2024, 10:10",TRUE
...,...,...,...,...,...,...
136,S476521A000764,Bit,FALSE,Mia sent you money via bit! 123 NIS are waitin...,"25/02/2024, 11:14",FALSE
137,S734689A482914,054-5172640,FALSE,Important: Tax refunds for 2022 are available!...,"08/01/2024, 08:50",TRUE
138,S409002A166504,Refunds,FALSE,"Nadav Numeric Algorithms, our records show you...","02/12/2024, 14:31",TRUE
139,S890123A456792,FastRefunds,FALSE,"Stav Combinatorics Partner, dont miss out! Emp...","30/01/2024, 18:00",TRUE


# Relational Optimization

In [290]:
# Every rename's schema can be propagated to the child node
rewrite(graph, lhs='y[op="rename",schema]->z[schema];x->y', p='x,z[schema]', rhs='x->z[schema={{y_schema}}]',
        render_rhs={'y_schema': lambda match: schema(match, 'y')}, is_recursive=True)
draw(graph,ret_mermaid=True)


flowchart TB
sms_rel["sms_rel
rel=#quot;sms_rel#quot;, rule_id={0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, #quot;fact#quot;}, schema=[#quot;sms_id#quot;, #quot;sender_id#quot;, #quot;is_sender_in_contacts#quot;, #quot;sms_body#quot;, #quot;sms_date#quot;, #quot;is_spam#quot;], op=#quot;get_rel#quot;, db=DB(sms_rel, sms_tax_year, sms_tax_link, sms_money, sms_wrong_receiver, sms_suspicious_sender, sms_tax), op_time=2.384185791015625e-06, final=True"]
sms_tax_year["sms_tax_year
rel=#quot;sms_tax_year#quot;, rule_id={0, 1, #quot;fact#quot;, 13}, op=#quot;union#quot;, schema=[#quot;sms_id#quot;, #quot;sender_id#quot;, #quot;is_sender_in_contacts#quot;, #quot;sms_body#quot;, #quot;sms_date#quot;, #quot;is_spam#quot;], op_time=0.0011000633239746094, final=True"]
1["1
op=#quot;project#quot;, schema=[#quot;sms_id#quot;, #quot;sender_id#quot;, #quot;is_sender_in_contacts#quot;, #quot;sms_body#quot;, #quot;sms_date#quot;, #quot;is_spam#quot;], rule_id={0}, op_time=0.0004482269287109375, final=Tru

In [291]:
# We start by removing all unnecesary project nodes - all projects here that are not childs of an product nodes are redundant
def is_not_product(match, node):
    if 'product' in match[node]['op']:
        return False
    return True

rewrite(graph, lhs='project_node[op="project", schema];x->project_node->y', p='x,y', rhs='x->y',condition=lambda match:
    is_not_product(match, 'x') and is_not_product(match, 'y'), is_recursive=True)

draw(graph,ret_mermaid=True)


flowchart TB
sms_rel["sms_rel
rel=#quot;sms_rel#quot;, rule_id={0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, #quot;fact#quot;}, schema=[#quot;sms_id#quot;, #quot;sender_id#quot;, #quot;is_sender_in_contacts#quot;, #quot;sms_body#quot;, #quot;sms_date#quot;, #quot;is_spam#quot;], op=#quot;get_rel#quot;, db=DB(sms_rel, sms_tax_year, sms_tax_link, sms_money, sms_wrong_receiver, sms_suspicious_sender, sms_tax), op_time=2.384185791015625e-06, final=True"]
sms_tax_year["sms_tax_year
rel=#quot;sms_tax_year#quot;, rule_id={0, 1, #quot;fact#quot;, 13}, op=#quot;union#quot;, schema=[#quot;sms_id#quot;, #quot;sender_id#quot;, #quot;is_sender_in_contacts#quot;, #quot;sms_body#quot;, #quot;sms_date#quot;, #quot;is_spam#quot;], op_time=0.0011000633239746094, final=True"]
2["2
op=#quot;project#quot;, schema=[#quot;sms_body#quot;], rule_id={0}, op_time=0.0004315376281738281, final=True"]
3["3
op=#quot;get_const#quot;, const_dict={#quot;_C0#quot;: #quot;refund for\\s*([1-2][0-9]{3})#quot;}, schema=[#quot

In [292]:
# Every rename's schema can be propagated to the child node
rewrite(graph, lhs='y[op="rename",schema]->z[schema];x->y', p='x,z[schema]', rhs='x->z[schema={{y_schema}}]',
        render_rhs={'y_schema': lambda match: schema(match, 'y')}, is_recursive=True)
draw(graph,ret_mermaid=True)


flowchart TB
sms_rel["sms_rel
rel=#quot;sms_rel#quot;, rule_id={0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, #quot;fact#quot;}, schema=[#quot;sms_id#quot;, #quot;sender_id#quot;, #quot;is_sender_in_contacts#quot;, #quot;sms_body#quot;, #quot;sms_date#quot;, #quot;is_spam#quot;], op=#quot;get_rel#quot;, db=DB(sms_rel, sms_tax_year, sms_tax_link, sms_money, sms_wrong_receiver, sms_suspicious_sender, sms_tax), op_time=2.384185791015625e-06, final=True"]
sms_tax_year["sms_tax_year
rel=#quot;sms_tax_year#quot;, rule_id={0, 1, #quot;fact#quot;, 13}, op=#quot;union#quot;, schema=[#quot;sms_id#quot;, #quot;sender_id#quot;, #quot;is_sender_in_contacts#quot;, #quot;sms_body#quot;, #quot;sms_date#quot;, #quot;is_spam#quot;], op_time=0.0011000633239746094, final=True"]
2["2
op=#quot;project#quot;, schema=[#quot;sms_body#quot;], rule_id={0}, op_time=0.0004315376281738281, final=True"]
3["3
op=#quot;get_const#quot;, const_dict={#quot;_C0#quot;: #quot;refund for\\s*([1-2][0-9]{3})#quot;}, schema=[#quot

In [293]:
def collection_schema(match, node):
    return match[node]['schema'][0]

rewrite(graph,
        lhs='union_node[op="union",schema];union_node->join_node[op="join",schema]->sms_rel[op="get_rel"],parent_union[op="union",schema]->union_node,join_node->ie_node[func,schema]',
        p='parent_union[op,schema],union_node[op],ie_node[func,schema],sms_rel[op]',
        rhs='parent_union[op,schema]->project_node[op="project",schema={{parent_union_schema}}]->join_node[op="join",schema={{join_node_schema}}]->union_node[op,schema={{ie_node_schema}}]->ie_node[func,schema], join_node->sms_rel[op]', 
        render_rhs={'ie_node_schema': lambda match: collection_schema(match, 'ie_node'), 'join_node_schema': lambda match: collection_schema(match, 'join_node'),
                    'parent_union_schema': lambda match: collection_schema(match, 'parent_union')},
        is_recursive=True)

draw(graph,ret_mermaid=True)


flowchart TB
sms_rel["sms_rel
rel=#quot;sms_rel#quot;, rule_id={0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, #quot;fact#quot;}, schema=[#quot;sms_id#quot;, #quot;sender_id#quot;, #quot;is_sender_in_contacts#quot;, #quot;sms_body#quot;, #quot;sms_date#quot;, #quot;is_spam#quot;], op=#quot;get_rel#quot;, db=DB(sms_rel, sms_tax_year, sms_tax_link, sms_money, sms_wrong_receiver, sms_suspicious_sender, sms_tax), op_time=2.384185791015625e-06, final=True"]
sms_tax_year["sms_tax_year
rel=#quot;sms_tax_year#quot;, rule_id={0, 1, #quot;fact#quot;, 13}, op=#quot;union#quot;, op_time=0.0011000633239746094, final=True, schema=[#quot;_F0#quot;, #quot;sms_body#quot;, #quot;res#quot;]"]
2["2
op=#quot;project#quot;, schema=[#quot;sms_body#quot;], rule_id={0}, op_time=0.0004315376281738281, final=True"]
3["3
op=#quot;get_const#quot;, const_dict={#quot;_C0#quot;: #quot;refund for\\s*([1-2][0-9]{3})#quot;}, schema=[#quot;_C0#quot;], rule_id={0}, op_time=0.0001964569091796875, final=True"]
4["4
op=#quot;pro

In [294]:

rewrite(graph, lhs='project_node[op="project",schema]->sms_rel[op="get_rel"]',
        p='sms_rel[op]',
        is_recursive=True)
rewrite(graph, lhs='sms_rel[op="get_rel"]',
        rhs='sms_rel[op],sms_body_project_node[op="project",schema={{sms_body}},sms_body]->sms_rel[op],\
                sender_id_project_node[op="project",schema={{sender_id}},sender_id]->sms_rel[op]',
        render_rhs={'sms_body': lambda match: ['sms_body'], 'sender_id': lambda match: ['sender_id']})   
rewrite(graph, lhs='sms_body[sms_body],product_node[op="product",schema]',
        rhs='product_node[op,schema]->sms_body[sms_body]',
        condition=lambda match: 'sms_body' in schema(match, 'product_node'))
rewrite(graph, lhs='sender_id[sender_id],product_node[op="product",schema]',
        rhs='product_node[op,schema]->sender_id[sender_id]',
        condition=lambda match: 'sender_id' in schema(match, 'product_node'))

draw(graph,ret_mermaid=True)


flowchart TB
sms_rel["sms_rel
rel=#quot;sms_rel#quot;, rule_id={0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, #quot;fact#quot;}, schema=[#quot;sms_id#quot;, #quot;sender_id#quot;, #quot;is_sender_in_contacts#quot;, #quot;sms_body#quot;, #quot;sms_date#quot;, #quot;is_spam#quot;], op=#quot;get_rel#quot;, db=DB(sms_rel, sms_tax_year, sms_tax_link, sms_money, sms_wrong_receiver, sms_suspicious_sender, sms_tax), op_time=2.384185791015625e-06, final=True"]
sms_tax_year["sms_tax_year
rel=#quot;sms_tax_year#quot;, rule_id={0, 1, #quot;fact#quot;, 13}, op=#quot;union#quot;, op_time=0.0011000633239746094, final=True, schema=[#quot;_F0#quot;, #quot;sms_body#quot;, #quot;res#quot;]"]
3["3
op=#quot;get_const#quot;, const_dict={#quot;_C0#quot;: #quot;refund for\\s*([1-2][0-9]{3})#quot;}, schema=[#quot;_C0#quot;], rule_id={0}, op_time=0.0001964569091796875, final=True"]
4["4
op=#quot;product#quot;, schema=[#quot;sms_body#quot;, #quot;_C0#quot;], rule_id={0}, op_time=0.0015482902526855469, final=True"]


In [295]:
graph.nodes['sms_rel']['db'] = DB({'sms_rel': sms_rel, 'sms_tax_year': pd.DataFrame()})
print("Profile data before optimization")
print(profile_df)
res, profile_data = compute_node(graph, root)
print("Profile data after optimization")
profile_df = pd.DataFrame(profile_data).T
print(profile_df)
res


Profile data before optimization
               row_count  total_time
get_rel         0.000000    0.000002
rename       4857.000000    0.006710
project     12943.000000    0.034812
get_const       0.000000    0.002115
product      3783.000000    0.020980
ie_map       3770.000000    0.258473
join         4103.000000    0.020842
union         879.000000    0.012318
total_time      0.361451    0.361451
Profile data after optimization
             row_count  total_time
get_rel        0.00000    0.000002
project     4809.00000    0.011746
get_const      0.00000    0.004007
product     3770.00000    0.034321
ie_map      3757.00000    0.281902
union        666.00000    0.007886
join        1719.00000    0.009558
total_time     0.35206    0.352060


,sms_id,sender_id,is_sender_in_contacts,sms_body,sms_date,is_spam
0,S567890A123457,FreeMoney,False,Nadav CS the tax refund for 2016 is about to e...,"31/10/2022, 13:14",True
1,S901234A567903,054-5172640,False,Important: Tax refunds for 2020 are available!...,"31/01/2024, 09:10",True
2,S567890A123460,054-5172640,False,See how much youre owed in a tax refund! Start...,"10/02/2024, 14:35",True
3,S352109A847629,TaxService,False,Maximize your tax refund in minutes! Click her...,"03/02/2024, 14:20",True
4,S123456A711123,TaxEasy,False,"Stav Compi, Your refund could be worth up to 2...","19/01/2024, 13:50",True
...,...,...,...,...,...,...
136,S567890A123454,052-7755016,False,"Following the economic situation, eligibility ...","29/10/2024, 12:03",False
137,S734689A482913,QuickTAXM,False,Workers from the last six years who rented? Yo...,"01/03/2022, 12:32",True
138,S901234A567901,EasyTaxRefund,False,"Stav Compi Rep, Last chance! Verify your eligi...","03/01/2024, 09:15",True
139,S582047B123198,rebar,False,"Hi Nadavi combi partner, Its time for a treat....","03/12/2024, 10:02",False


# Subregex optimization

In [296]:
# First, propagate the rel attribute to great grandchildren nodes - the product nodes
rewrite(graph, lhs='x[rel]->_->_->y', rhs='x[rel],y[rel={{rel}}]', condition=lambda match:match['x']['rel']!='sms_tax', render_rhs={'rel':lambda match: match['x']['rel']})
rewrite(graph, lhs='x[rel]->y[const_dict]',p='x->y[const_dict]',
        condition=lambda match: match['y']['const_dict'] == {'_C0':'([nN][iI][sS])'})
g = graph.copy()

draw(graph,ret_mermaid=True)


flowchart TB
sms_rel["sms_rel
rel=#quot;sms_rel#quot;, rule_id={0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, #quot;fact#quot;}, schema=[#quot;sms_id#quot;, #quot;sender_id#quot;, #quot;is_sender_in_contacts#quot;, #quot;sms_body#quot;, #quot;sms_date#quot;, #quot;is_spam#quot;], op=#quot;get_rel#quot;, db=DB(sms_rel, sms_tax_year), op_time=1.6689300537109375e-06, final=True"]
sms_tax_year["sms_tax_year
rel=#quot;sms_tax_year#quot;, rule_id={0, 1, #quot;fact#quot;, 13}, op=#quot;union#quot;, op_time=0.0006728172302246094, final=True, schema=[#quot;_F0#quot;, #quot;sms_body#quot;, #quot;res#quot;]"]
3["3
op=#quot;get_const#quot;, const_dict={#quot;_C0#quot;: #quot;refund for\\s*([1-2][0-9]{3})#quot;}, schema=[#quot;_C0#quot;], rule_id={0}, op_time=0.00021123886108398438, final=True"]
4["4
op=#quot;product#quot;, schema=[#quot;sms_body#quot;, #quot;_C0#quot;], rule_id={0}, op_time=0.0017361640930175781, final=True, rel=#quot;sms_tax_year#quot;"]
5["5
op=#quot;project#quot;, schema=[#quot;_C

In [ ]:
def largest_subregex_rewrite(g):
    subregex_dict = {'sms_tax_year': {"_C0":'[1-2][0-9]{3}'},
                     'sms_tax_link': {"_C0":'http.*\.com'},
                     'sms_money': {"_C0":'\d{1,3}'},
                     'sms_wrong_receiver': {"_C0":'Stav My Love|Stav CS Technion|Stav Combinatorics Partner|Stav Aviram - Army|Stav Compi|Stav Compi Rep'}
    }
    
    for regex in subregex_dict:
        rewrite(g, lhs='project_node[op="project"];product_node[op="product",rel="'+regex+'"]->project_node',
        p='product_node[op,rel],project_node[op]',
        rhs='new_ie_node[op="ie_map",func={{func}},in_arity=2,out_arity=1,schema={{ie_schema}},name="rgx_is_match",in_schema={{in_schema}},out_schema={{out_schema}}]\
            ,new_product_node->new_get_const_node[op="get_const",const_dict={{const_dict}},schema={{const_schema}}],\
            new_product_node[op="product",schema={{product_schema}}]->project_node[op],\
            new_project_node_2[op="project",schema={{project_schema_2}}]->new_rename_node[op="rename",schema={{ie_schema}}]->new_ie_node\
            ->new_project_node_1[op="project",schema={{project_schema_1}}]->new_product_node,\
            product_node[op,rel]->new_project_node_2',
        render_rhs={'func': lambda match: rgx_is_match,
                    'ie_schema': lambda match: ['_F0', 'sms_body', 'res'],
                    'product_schema': lambda match: ['sms_body', '_C0'],
                    'const_schema': lambda match: ['_C0'],
                    'project_schema_1': lambda match: ['_C0', 'sms_body'],
                    'project_schema_2': lambda match: ['sms_body'],
                    'const_dict': lambda match: subregex_dict[regex],
                    'in_schema': lambda match: [str,str],
                    'out_schema': lambda match: [str]})

In [298]:
largest_subregex_rewrite(graph)
draw(graph,ret_mermaid=True)


flowchart TB
sms_rel["sms_rel
rel=#quot;sms_rel#quot;, rule_id={0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, #quot;fact#quot;}, schema=[#quot;sms_id#quot;, #quot;sender_id#quot;, #quot;is_sender_in_contacts#quot;, #quot;sms_body#quot;, #quot;sms_date#quot;, #quot;is_spam#quot;], op=#quot;get_rel#quot;, db=DB(sms_rel, sms_tax_year), op_time=1.6689300537109375e-06, final=True"]
sms_tax_year["sms_tax_year
rel=#quot;sms_tax_year#quot;, rule_id={0, 1, #quot;fact#quot;, 13}, op=#quot;union#quot;, op_time=0.0006728172302246094, final=True, schema=[#quot;_F0#quot;, #quot;sms_body#quot;, #quot;res#quot;]"]
3["3
op=#quot;get_const#quot;, const_dict={#quot;_C0#quot;: #quot;refund for\\s*([1-2][0-9]{3})#quot;}, schema=[#quot;_C0#quot;], rule_id={0}, op_time=0.00021123886108398438, final=True"]
4["4
op=#quot;product#quot;, schema=[#quot;sms_body#quot;, #quot;_C0#quot;], rule_id={0}, op_time=0.0017361640930175781, final=True, rel=#quot;sms_tax_year#quot;"]
5["5
op=#quot;project#quot;, schema=[#quot;_C

In [ ]:
graph.nodes['sms_rel']['db'] = DB({'sms_rel': sms_rel, 'sms_tax': pd.DataFrame()})
print("Profile data before optimization")
print(profile_df)
res, profile_data = compute_node(graph, root)
print("Profile data after optimization")
profile_df = pd.DataFrame(profile_data).T
print(profile_df)
res

Profile data before optimization
             row_count  total_time
get_rel        0.00000    0.000002
project     4809.00000    0.011746
get_const      0.00000    0.004007
product     3770.00000    0.034321
ie_map      3757.00000    0.281902
union        666.00000    0.007886
join        1719.00000    0.009558
total_time     0.35206    0.352060
Profile data after optimization
              row_count  total_time
get_rel        0.000000    0.000056
project     4899.000000    0.042146
get_const      0.000000    0.008825
product     3365.000000    0.054127
ie_map      3348.000000    0.260607
rename       499.000000    0.000695
union        666.000000    0.007969
join        1719.000000    0.009189
total_time     0.393981    0.393981


,sms_id,sender_id,is_sender_in_contacts,sms_body,sms_date,is_spam
0,S567890A123457,FreeMoney,False,Nadav CS the tax refund for 2016 is about to e...,"31/10/2022, 13:14",True
1,S901234A567903,054-5172640,False,Important: Tax refunds for 2020 are available!...,"31/01/2024, 09:10",True
2,S567890A123460,054-5172640,False,See how much youre owed in a tax refund! Start...,"10/02/2024, 14:35",True
3,S352109A847629,TaxService,False,Maximize your tax refund in minutes! Click her...,"03/02/2024, 14:20",True
4,S123456A711123,TaxEasy,False,"Stav Compi, Your refund could be worth up to 2...","19/01/2024, 13:50",True
...,...,...,...,...,...,...
136,S567890A123454,052-7755016,False,"Following the economic situation, eligibility ...","29/10/2024, 12:03",False
137,S734689A482913,QuickTAXM,False,Workers from the last six years who rented? Yo...,"01/03/2022, 12:32",True
138,S901234A567901,EasyTaxRefund,False,"Stav Compi Rep, Last chance! Verify your eligi...","03/01/2024, 09:15",True
139,S582047B123198,rebar,False,"Hi Nadavi combi partner, Its time for a treat....","03/12/2024, 10:02",False
